# docuchat demo

End-to-end walkthrough: ingest a PDF, classify pages, build the store, and ask questions with citations.

**Requires `ANTHROPIC_API_KEY` to be set in the environment** -- `ask()` calls a real Anthropic LLM.

In [ ]:
from docuchat.config import Settings
from docuchat.ingest import load_pdf
from docuchat.classify import classify_pages
from docuchat.index import build_store
from docuchat.models import get_llm
from docuchat.pipeline import ask

## 1. Configure and ingest the fixture PDF

In [ ]:
cfg = Settings()
pages = load_pdf("../tests/fixtures/cfpb_closing_disclosure.pdf", cfg)
len(pages)

## 2. Classify pages (heuristic first, LLM fallback)

In [ ]:
pages = classify_pages(pages, get_llm(cfg), cfg)
pages[0]["doc_type"]

## 3. Build the hybrid (vector + BM25) store

In [ ]:
store = build_store(pages, cfg)
len(store.nodes)

## 4. Page and chunk breakdown

In [ ]:
from collections import Counter

print(f"Pages: {len(pages)}")
print(f"Chunks: {len(store.nodes)}")
print("doc_type breakdown:", Counter(p["doc_type"] for p in pages))

## 5. Ask questions

In [ ]:
questions = [
    "What is the loan amount?",
    "What are the closing costs?",
    "Is there a prepayment penalty?",
]

for q in questions:
    result = ask(q, store, cfg)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    for src in result["sources"]:
        print(f"  - {src['filename']} p.{src['page_number']} ({src['doc_type']})")
    print()